# Revisione volti per fascia d'età
Legge `output/results.csv` e `no_faces.csv` e mostra un mosaico per ogni fascia, con i volti a confidenza più bassa per primi (i più probabili errori). Basta eseguire tutte le celle: nessun widget.

In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd
from PIL import Image
from IPython.display import display

OUT = Path("../output")     # cartella con results.csv
THUMB = 128                 # lato miniatura (px)
COLS = 10                   # miniature per riga
MAX_PER_BIN = 200           # volti mostrati per fascia (None = tutti)
ORDER_ASC = True            # True: confidenza crescente (dubbi per primi)

df = pd.read_csv(OUT / "results.csv")
df["age_bin"] = df["age_bin"].astype(str)
bins = sorted(df["age_bin"].unique(), key=lambda b: int("".join(c for c in b.split("-")[0].split()[-1] if c.isdigit()) or 0))
display(df.groupby("age_bin").agg(n=("file", "size"), conf_media=("age_conf", "mean"), conf_min=("age_conf", "min")).loc[bins])

In [ ]:
def thumb(r):
    img = cv2.imread(r.file)
    c = img[r.y1:r.y2, r.x1:r.x2]
    c = cv2.resize(c, (THUMB, THUMB))
    cv2.rectangle(c, (0, 0), (THUMB, 16), (0, 0, 0), -1)
    cv2.putText(c, f"{r.age_conf:.0%}", (3, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
    return c

def mosaic(sub):
    tiles = [thumb(r) for r in sub.itertuples()]
    while len(tiles) % COLS: tiles.append(np.zeros((THUMB, THUMB, 3), np.uint8))
    rows = [np.hstack(tiles[i:i + COLS]) for i in range(0, len(tiles), COLS)]
    return Image.fromarray(cv2.cvtColor(np.vstack(rows), cv2.COLOR_BGR2RGB))

for b in bins:
    sub = df[df.age_bin == b].sort_values("age_conf", ascending=ORDER_ASC)
    shown = sub if MAX_PER_BIN is None else sub.head(MAX_PER_BIN)
    print(f"=== {b}: {len(sub)} volti (mostrati {len(shown)}) ===")
    display(mosaic(shown))

## Immagini senza volti rilevati (possibili falsi negativi del detector)
Se contengono un volto, abbassa `--det-score` e rilancia.

In [ ]:
nf_path = OUT / "no_faces.csv"
nf = pd.read_csv(nf_path) if nf_path.exists() else pd.DataFrame(columns=["file"])
print(f"{len(nf)} immagini senza volti")
if len(nf):
    tiles = []
    for f in nf.file.head(MAX_PER_BIN or None):
        t = cv2.resize(cv2.imread(f), (THUMB, THUMB))
        tiles.append(t)
    while len(tiles) % COLS: tiles.append(np.zeros((THUMB, THUMB, 3), np.uint8))
    rows = [np.hstack(tiles[i:i + COLS]) for i in range(0, len(tiles), COLS)]
    display(Image.fromarray(cv2.cvtColor(np.vstack(rows), cv2.COLOR_BGR2RGB)))